# SKRIPSI KEMAL RAMADHAN
## SISTEM PENDUKUNG KEPUTUSAN MITIGASI RISIKO PROYEK TEKNOLOGI INFORMASI DAN RISET & PENGEMBANGAN MENGGUNAKAN ASSOCIATION RULE MINING DAN ANALYTICAL HIERARCHY PROCESS

## Uji Coba Dipisah Project IT & RND dan Dipisah Tiap Level Risiko

## Import Library

In [17]:
import pandas as pd
import numpy as np
import datetime
import optuna
import os
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
from collections import Counter

# 1. Asscoiation Rule Mining

## Data Pre-Processing
### Data Cleaning
Menghapus kolom tidak relevan (project_id) dan handling missing values (cek dan imputasi)

In [26]:
# load dataset
print("Memuat Dataset dan Filtering...")
df_raw = pd.read_csv('DATASET/project_risk_raw_dataset.csv')

# filter for IT and R&D project
df = df_raw[df_raw['Project_Type'].isin(['IT', 'R&D'])].copy()
print(f"   -> Sisa data setelah filter IT & R&D: {df.shape[0]} baris.")

Memuat Dataset dan Filtering...
   -> Sisa data setelah filter IT & R&D: 1969 baris.


### Menghapus kolom "project_id"

In [27]:
# deleting "project_id" column
print("Data Cleaning (Menghapus Project_ID)")
if 'Project_ID' in df.columns:
    df = df.drop(columns=['Project_ID'])

def fix_excel_time_to_float(val):
    if isinstance(val, str) and ':' in val:
        try:
            parts = val.split(':')
            return float(f"{parts[0]}.{parts[1]}")
        except:
            return val
    return val

# Terapkan pembersihan HANYA untuk kolom object/teks
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(fix_excel_time_to_float)
    df[col] = pd.to_numeric(df[col], errors='ignore')

Data Cleaning (Menghapus Project_ID)


/var/folders/ys/km4g4xjx66zbg88w7ddg59tw0000gn/T/ipykernel_1110/2879191401.py:18: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


### Cek Missing Value
Jumlah baris dengan missing value, jumlah kolom dengan missing value di tiap barisnya, dan tipe data kolom yang memiliki missing value.

In [28]:
# Hitung jumlah nilai kosong (missing values) di setiap baris (record)
missing_per_row = df.isnull().sum(axis=1)
total_cols = df.shape[1]

# Ambil hanya baris yang memiliki minimal 1 missing value
baris_dengan_missing = missing_per_row[missing_per_row > 0]

# Tampilkan detailnya
print(f"   -> Ditemukan {len(baris_dengan_missing)} baris yang memiliki missing value.")

if not baris_dengan_missing.empty:
    # Menghitung jumlah paling sedikit dan paling banyak
    min_missing = baris_dengan_missing.min()
    max_missing = baris_dengan_missing.max()
    
    print(f"   -> Jumlah missing value PALING SEDIKIT dalam satu baris : {min_missing} kolom")
    print(f"   -> Jumlah missing value PALING BANYAK dalam satu baris   : {max_missing} kolom")
    
    print("\n   -> RINCIAN KOLOM YANG MENGALAMI MISSING VALUE:")
    
    # Menghitung jumlah missing value per kolom
    missing_per_col = df.isnull().sum()
    
    # Filter hanya kolom yang punya missing value (> 0)
    kolom_dengan_missing = missing_per_col[missing_per_col > 0]
    
    for nama_kolom, jumlah_kosong in kolom_dengan_missing.items():
        tipe_data = df[nama_kolom].dtype
        print(f"      • {nama_kolom}")
        print(f"        - Jumlah kosong : {jumlah_kosong} baris")
        print(f"        - Tipe Data     : {tipe_data}")
    # =========================================================================
    
    print("\n   -> Detail jumlah missing value per baris (Menampilkan 15 baris pertama):")
    print(baris_dengan_missing.head(15))

   -> Ditemukan 968 baris yang memiliki missing value.
   -> Jumlah missing value PALING SEDIKIT dalam satu baris : 1 kolom
   -> Jumlah missing value PALING BANYAK dalam satu baris   : 3 kolom

   -> RINCIAN KOLOM YANG MENGALAMI MISSING VALUE:
      • Tech_Environment_Stability
        - Jumlah kosong : 588 baris
        - Tipe Data     : object
      • Change_Control_Maturity
        - Jumlah kosong : 353 baris
        - Tipe Data     : object
      • Risk_Management_Maturity
        - Jumlah kosong : 354 baris
        - Tipe Data     : object

   -> Detail jumlah missing value per baris (Menampilkan 15 baris pertama):
6     2
17    1
24    3
25    1
26    1
27    2
29    1
31    1
36    2
38    2
39    1
44    2
45    1
53    1
57    1
dtype: int64


In [29]:
# model-based imputation untuk mengisi missing value (KNN)

print("Menjalankan KNN Imputation untuk sisa missing values...")

# CEK SEBELUM IMPUTASI
total_missing_sebelum = df.isnull().sum().sum()
print(f"   [!] Total sel kosong SEBELUM imputasi : {total_missing_sebelum} sel data")

df_imputed = df.copy()
label_encoders = {}
kolom_kategorik = df_imputed.select_dtypes(include=['object']).columns

for col in kolom_kategorik:
    le = LabelEncoder()
    
    # Cari baris mana saja yang tidak kosong (bukan missing value)
    mask_notnull = df_imputed[col].notnull()
    
    # Paksa semua data yang tidak kosong tersebut menjadi format teks (String)
    df_imputed.loc[mask_notnull, col] = df_imputed.loc[mask_notnull, col].astype(str)
    
    # Ambil data yang sudah seragam formatnya
    not_null_data = df_imputed.loc[mask_notnull, col]
    
    # Fit dan Transform menggunakan LabelEncoder
    le.fit(not_null_data)
    df_imputed.loc[mask_notnull, col] = le.transform(not_null_data)
    
    # Simpan encoder untuk proses pengembalian teks nanti
    label_encoders[col] = le

# Eksekusi KNN Imputer (berdasarkan kedekatan pola antar-proyek)
imputer = KNNImputer(n_neighbors=5)
df_imputed_array = imputer.fit_transform(df_imputed)
df_imputed = pd.DataFrame(df_imputed_array, columns=df.columns, index=df.index)

# Kembalikan data angka tadi menjadi teks kategori asli
for col, le in label_encoders.items():
    # Bulatkan hasil regresi KNN ke integer terdekat agar cocok dengan indeks label
    df_imputed[col] = df_imputed[col].round().astype(int)
    # Pastikan hasil prediksi tidak melewati batas indeks array kelas yang ada
    df_imputed[col] = df_imputed[col].clip(0, len(le.classes_) - 1)
    # Kembalikan angka menjadi teks asli
    df_imputed[col] = le.inverse_transform(df_imputed[col])

df = df_imputed.copy()

# CEK SESUDAH IMPUTASI
total_missing_sesudah = df.isnull().sum().sum()
print(f"   [!] Total sel kosong SESUDAH imputasi : {total_missing_sesudah} sel data")
print("   -> Status: Missing values berhasil diimputasi sepenuhnya")

Menjalankan KNN Imputation untuk sisa missing values...
   [!] Total sel kosong SEBELUM imputasi : 1295 sel data
   [!] Total sel kosong SESUDAH imputasi : 0 sel data
   -> Status: Missing values berhasil diimputasi sepenuhnya


### Data Discretization (Equal-Frequency Binning)
Mengubah data numerik menjadi kategorik dengan membaginya ke dalam 3 kategori (low, medium, high) dengan jumlah data yang seimbang di setiap kategori,

In [30]:
print("Diskritisasi Data Numerik (Equal-Frequency Binning)...")
kolom_numerik = df.select_dtypes(include=['int64', 'float64']).columns

for col in kolom_numerik:
    try:
        # qcut membagi distribusi frekuensi sama rata menjadi 3 (Low, Medium, High)
        labels = [f'{col}_Low', f'{col}_Medium', f'{col}_High']
        df[col] = pd.qcut(df[col], q=3, labels=labels, duplicates='drop')
        
        # cek hasil distribusi
        print(f"\nDistribusi hasil binning untuk kolom: {col}")
        print(df[col].value_counts())
        
    except ValueError:
        # Jika nilai terlalu seragam (outliers statis), ubah ke string agar tidak error
        df[col] = df[col].astype(str)

Diskritisasi Data Numerik (Equal-Frequency Binning)...

Distribusi hasil binning untuk kolom: Team_Size
Team_Size
Team_Size_Low       745
Team_Size_High      628
Team_Size_Medium    596
Name: count, dtype: int64

Distribusi hasil binning untuk kolom: Project_Budget_USD
Project_Budget_USD
Project_Budget_USD_Low       657
Project_Budget_USD_Medium    656
Project_Budget_USD_High      656
Name: count, dtype: int64

Distribusi hasil binning untuk kolom: Estimated_Timeline_Months
Estimated_Timeline_Months
Estimated_Timeline_Months_Low       695
Estimated_Timeline_Months_High      645
Estimated_Timeline_Months_Medium    629
Name: count, dtype: int64

Distribusi hasil binning untuk kolom: Complexity_Score
Complexity_Score
Complexity_Score_Low       657
Complexity_Score_High      657
Complexity_Score_Medium    655
Name: count, dtype: int64

Distribusi hasil binning untuk kolom: Stakeholder_Count
Stakeholder_Count
Stakeholder_Count_Low       750
Stakeholder_Count_High      650
Stakeholder_Count_

### One-Hot Encoding
Mengubah data menjadi format boolean untuk dimasukkan ke dalam algoritma association rule mining.

In [31]:
# One-Hot Encoding untuk Algoritma FP-Growth
df_encoded = pd.get_dummies(df, dtype=bool)
print(f"   -> Data siap. Total dimensi setelah encoding: {df_encoded.shape[1]} kolom.")

   -> Data siap. Total dimensi setelah encoding: 208 kolom.


### FP-Growth & Rule Generation

#### Uji Coba Nilai Support dan Confidence Hasil Dari PSO-ARM
Support: 6,1% | Confidence: 50%

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth

print("Memulai Looping Eksperimen FP-Growth Spesifik per Tipe Proyek (IT & R&D)...")

# Asumsi nama kolom target sesuai dengan dataset yang sudah di One-Hot Encoding
target_risks = ['Risk_Level_Low', 'Risk_Level_Medium', 'Risk_Level_High', 'Risk_Level_Critical']

# Asumsi nama kolom tipe proyek hasil One-Hot Encoding
# Sesuaikan string ini jika prefix get_dummies-nya berbeda di script lu
project_types = ['Project_Type_IT', 'Project_Type_R&D'] 

# Parameter ambang batas sesuai target (Support 6.1%, Confidence 50.00%)
support_levels = [0.0610]
confidence_levels = [0.5000]

# LOOPING LEVEL 1: BERDASARKAN TIPE PROYEK
for p_type in project_types:
    tipe_nama = p_type.replace('Project_Type_', '').replace('&', 'n') # Membersihkan nama untuk string/file
    
    print("\n" + "#"*80)
    print(f" MENGANALISIS TIPE PROYEK: {p_type.replace('Project_Type_', '')}")
    print("#"*80)

    # Mengecek apakah kolom tipe proyek ada di dataset
    if p_type not in df_encoded.columns:
        print(f"  [X] Kolom {p_type} tidak ditemukan di df_encoded. Pastikan nama hasil encoding benar. Lewati.")
        continue

    # 1. Partisi Ekosistem Data: Ambil HANYA baris proyek milik tipe ini
    df_project = df_encoded[df_encoded[p_type] == True].copy()
    
    # 2. Buang semua kolom yang berbau 'Project_Type' agar tidak muncul berulang-ulang di dalam Rules
    kolom_tipe_proyek = [col for col in df_encoded.columns if 'Project_Type' in col]
    df_project = df_project.drop(columns=kolom_tipe_proyek, errors='ignore')

    # 3. Hitung total seluruh populasi data KHUSUS untuk tipe proyek ini (sebagai Data Global Baru)
    total_global_data = len(df_project)
    print(f"Total populasi data {tipe_nama}: {total_global_data} proyek.")

    if total_global_data == 0:
        print("  [!] Data kosong. Lewati.")
        continue

    # LOOPING LEVEL 2 & 3: PARAMETER SUPPORT & CONFIDENCE
    for MIN_SUPPORT in support_levels:
        for MIN_CONFIDENCE in confidence_levels:
            print("\n" + "="*70)
            print(f" EKSPERIMEN [{tipe_nama}]: Support {MIN_SUPPORT*100:.2f}% | Confidence {MIN_CONFIDENCE*100:.2f}%")
            print("="*70)
            
            master_rules_list = []
            
            # Dictionary untuk menyimpan jumlah aturan tiap level risiko
            rekap_aturan = {target: 0 for target in target_risks}

            # LOOPING LEVEL 4: BERDASARKAN TARGET LEVEL RISIKO
            for target in target_risks:
                if target not in df_project.columns:
                    continue
                    
                print(f"\n--- Menganalisis Partisi: {target} ---")
                
                # Partisi data: Ambil HANYA proyek yang risikonya sesuai target di dalam ekosistem tipe proyek ini
                df_partisi = df_project[df_project[target] == True].copy()
                
                # Buang kolom tingkat risiko dari data partisi agar algoritma murni mencari (Antecedents)
                df_partisi_murni = df_partisi.drop(columns=target_risks, errors='ignore')
                
                print(f"   -> Mencari pola lokal pada {len(df_partisi)} kasus {target}...")
                
                if len(df_partisi) == 0:
                    continue
                    
                # FP-Growth pada data partisi lokal (menggunakan minimal support)
                frequent_itemsets = fpgrowth(df_partisi_murni, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
                
                if frequent_itemsets.empty:
                    print("   -> Tidak ada pola penyebab yang memenuhi nilai minimum support.")
                    continue
                
                print(f"   -> Ditemukan {len(frequent_itemsets)} kombinasi potensial. Mengevaluasi ke seluruh populasi {tipe_nama}...")
                
                # Support probabilitas alami dari target di level populasi tipe proyek ini
                support_target_global = df_project[target].sum() / total_global_data
                
                # Rule Generation (Cross-Check kandidat ke Data Global Tipe Proyek)
                aturan_valid = 0
                for idx, row in frequent_itemsets.iterrows():
                    # Ambil kombinasi penyebab
                    antecedents = list(row['itemsets'])
                    
                    # Hitung seberapa sering penyebab ini muncul di keseluruhan data tipe proyek ini
                    mask_antecedents_global = df_project[antecedents].all(axis=1)
                    support_antecedent_global = mask_antecedents_global.sum() / total_global_data
                    
                    # Hitung seberapa sering penyebab ini terjadi BERSAMAAN dengan target
                    mask_keduanya_global = mask_antecedents_global & df_project[target]
                    support_keduanya_global = mask_keduanya_global.sum() / total_global_data
                    
                    # Hindari pembagian dengan nol
                    if support_antecedent_global > 0:
                        # Hitung metrik asli secara matematis
                        asli_confidence = support_keduanya_global / support_antecedent_global
                        asli_lift = asli_confidence / support_target_global
                        
                        # Saring berdasarkan MIN_CONFIDENCE yang ditetapkan (0.50)
                        if asli_confidence >= MIN_CONFIDENCE:
                            aturan_valid += 1
                            master_rules_list.append({
                                'Tipe_Proyek': tipe_nama,
                                'Antecedents': ', '.join(antecedents),
                                'Consequents': target,
                                'Support': support_keduanya_global,
                                'Confidence': asli_confidence,
                                'Lift': asli_lift
                            })
                            
                print(f"   -> Lolos evaluasi: {aturan_valid} aturan yang kuat dan valid.")
                
                # Simpan jumlah aturan valid ke dalam dictionary rekap
                rekap_aturan[target] = aturan_valid

            # ==========================================
            # HASIL AKHIR UNTUK TIAP KOMBINASI TIPE PROYEK
            # ==========================================
            master_rules = pd.DataFrame(master_rules_list)
            
            # Menampilkan ringkasan sebelum tabel
            print("\n" + "-"*50)
            print(f"RINGKASAN JUMLAH ATURAN ({tipe_nama}):")
            for tgt, jumlah in rekap_aturan.items():
                print(f"   • {tgt} : {jumlah} aturan")
            print("-"*50)

            if not master_rules.empty:
                # Urutkan prioritas berdasarkan Target Risiko, lalu Lift terkuat
                master_rules = master_rules.sort_values(by=['Consequents', 'Lift'], ascending=[True, False])
                
                print(f"\n PROSES SELESAI! Total keseluruhan aturan divalidasi untuk {tipe_nama}: {len(master_rules)} aturan.")
                
                # Merapikan hasil desimal agar mudah dibaca
                master_rules['Support'] = master_rules['Support'].round(4)
                master_rules['Confidence'] = master_rules['Confidence'].round(4)
                master_rules['Lift'] = master_rules['Lift'].round(4)
                
                # Menampilkan 3 teratas
                print(f"\n 3 Aturan Terbaik dari Proyek {tipe_nama}:")
                display(master_rules.head(3))
                
                # Penamaan file CSV otomatis & aman
                str_sup = str(MIN_SUPPORT).replace('.', '_')
                str_conf = str(MIN_CONFIDENCE).replace('.', '_')
                nama_file = f'hasil_fpgrowth_{tipe_nama}_sup_{str_sup}_conf_{str_conf}.csv'
                
                master_rules.to_csv(nama_file, index=False)
                print(f" [✓] File berhasil disimpan sebagai: '{nama_file}'")
            else:
                print(f"\n [X] Proses selesai. Tidak ada aturan yang lolos uji evaluasi untuk proyek {tipe_nama}.")
            
print("\n" + "="*80)
print(" SEMUA EKSPERIMEN (IT & R&D) SELESAI! Silakan cek direktori untuk melihat file CSV-nya.")
print("="*80)

Memulai Looping Eksperimen FP-Growth Spesifik per Tipe Proyek (IT & R&D)...

################################################################################
 MENGANALISIS TIPE PROYEK: IT
################################################################################
Total populasi data IT: 1381 proyek.

 EKSPERIMEN [IT]: Support 6.10% | Confidence 50.00%

--- Menganalisis Partisi: Risk_Level_Low ---
   -> Mencari pola lokal pada 305 kasus Risk_Level_Low...
   -> Ditemukan 64111 kombinasi potensial. Mengevaluasi ke seluruh populasi IT...
   -> Lolos evaluasi: 3406 aturan yang kuat dan valid.

--- Menganalisis Partisi: Risk_Level_Medium ---
   -> Mencari pola lokal pada 525 kasus Risk_Level_Medium...
   -> Ditemukan 49849 kombinasi potensial. Mengevaluasi ke seluruh populasi IT...
   -> Lolos evaluasi: 4768 aturan yang kuat dan valid.

--- Menganalisis Partisi: Risk_Level_High ---
   -> Mencari pola lokal pada 313 kasus Risk_Level_High...
   -> Ditemukan 47557 kombinasi potensial. Meng

,Tipe_Proyek,Antecedents,Consequents,Support,Confidence,Lift
8965,IT,External_Dependencies_Count_External_Dependenc...,Risk_Level_Critical,0.0116,1.0000,5.8025
10367,IT,"Tech_Environment_Stability_Legacy/Unstable, Da...",Risk_Level_Critical,0.0181,1.0000,5.8025
8614,IT,Estimated_Timeline_Months_Estimated_Timeline_M...,Risk_Level_Critical,0.0232,0.9412,5.4612


 [✓] File berhasil disimpan sebagai: 'hasil_fpgrowth_IT_sup_0_061_conf_0_5.csv'

################################################################################
 MENGANALISIS TIPE PROYEK: R&D
################################################################################
Total populasi data RnD: 588 proyek.

 EKSPERIMEN [RnD]: Support 6.10% | Confidence 50.00%

--- Menganalisis Partisi: Risk_Level_Low ---
   -> Mencari pola lokal pada 38 kasus Risk_Level_Low...
   -> Ditemukan 86359 kombinasi potensial. Mengevaluasi ke seluruh populasi RnD...
   -> Lolos evaluasi: 5012 aturan yang kuat dan valid.

--- Menganalisis Partisi: Risk_Level_Medium ---
   -> Mencari pola lokal pada 143 kasus Risk_Level_Medium...
   -> Ditemukan 81304 kombinasi potensial. Mengevaluasi ke seluruh populasi RnD...
   -> Lolos evaluasi: 3704 aturan yang kuat dan valid.

--- Menganalisis Partisi: Risk_Level_High ---
   -> Mencari pola lokal pada 196 kasus Risk_Level_High...
   -> Ditemukan 81636 kombinasi potensia

,Tipe_Proyek,Antecedents,Consequents,Support,Confidence,Lift
22309,RnD,"Org_Process_Maturity_Ad-hoc, Team_Colocation_H...",Risk_Level_Critical,0.0221,1.0,2.7867
28169,RnD,Previous_Delivery_Success_Rate_Previous_Delive...,Risk_Level_Critical,0.0289,1.0,2.7867
29845,RnD,"Org_Process_Maturity_Ad-hoc, Past_Similar_Proj...",Risk_Level_Critical,0.0255,1.0,2.7867


 [✓] File berhasil disimpan sebagai: 'hasil_fpgrowth_RnD_sup_0_061_conf_0_5.csv'

 SEMUA EKSPERIMEN (IT & R&D) SELESAI! Silakan cek direktori untuk melihat file CSV-nya.


# 2. Seleksi Kriteria
Aturan dengan lift>1 diurutkan untuk kemudian dipotong tepat sebelum terjadi penurunan nilai lift paling drastis (selisih terbesar). Kemudian mengekstraksi faktor risiko paling dominan.

### Revisi untuk mengatasi penurunan nilai lift signifikan di data pertama atau kedua (menyebabkan kriteria dominan berjumlah <9)

In [33]:
print("="*80)
print("MEMULAI EKSEKUSI PEMOTONGAN DINAMIS & EKSTRAKSI KRITERIA (IT & R&D)")
print("="*80)

# 1. BACA DAN GABUNGKAN KEDUA FILE CSV
file_list = [
    'hasil_fpgrowth_IT_sup_0_061_conf_0_5.csv', 
    'hasil_fpgrowth_RnD_sup_0_061_conf_0_5.csv'
]

df_list = []
for file in file_list:
    if os.path.exists(file):
        df_temp = pd.read_csv(file)
        df_list.append(df_temp)
        print(f" [✓] File dimuat: {file} ({len(df_temp)} aturan)")
    else:
        print(f" [X] File tidak ditemukan: {file}")

if not df_list:
    print("Tidak ada file yang bisa diproses. Program dihentikan.")
    exit()

# Gabungkan kedua data menjadi satu Dataframe Master
df_rules = pd.concat(df_list, ignore_index=True)

# 2. PEMBERSIHAN DATA
df_rules['Lift'] = pd.to_numeric(df_rules['Lift'], errors='coerce')
df_rules = df_rules.dropna(subset=['Lift'])
df_rules = df_rules[df_rules['Lift'] > 1.0].copy()

tipe_proyek_list = df_rules['Tipe_Proyek'].unique()

# ==========================================
# 3. LOOPING EKSTRAKSI & PEMOTONGAN DINAMIS
# ==========================================

for tipe in tipe_proyek_list:
    print("\n" + "#"*70)
    print(f" MENGANALISIS DOMAIN: PROYEK {tipe.upper()}")
    print("#"*70)
    
    df_tipe = df_rules[df_rules['Tipe_Proyek'] == tipe].copy()
    target_risks = df_tipe['Consequents'].unique()

    for target in target_risks:
        print(f"\n" + "-"*50)
        print(f" Target Risiko: {target}")
        print("-" * 50)
        
        df_target = df_tipe[df_tipe['Consequents'] == target].copy()
        
        if df_target.empty:
            print("   -> Tidak ada aturan untuk target risiko ini.")
            continue
            
        # Urutkan Menurun (Descending) berdasarkan Lift
        df_sorted = df_target.sort_values(by='Lift', ascending=False).reset_index(drop=True)
        
        # LOGIKA PEMOTONGAN DINAMIS DENGAN SYARAT MINIMAL 9 KRITERIA
        if target == 'Risk_Level_High':
            print("   -> (BYPASS KHUSUS) Target Risk_Level_High terdeteksi.")
            print("   -> Mengambil seluruh aturan yang tersedia tanpa pemotongan.")
            df_selected = df_sorted.copy()
            
        elif len(df_sorted) > 1:
            # Hitung selisih penurunan nilai Lift
            df_sorted['lift_berikutnya'] = df_sorted['Lift'].shift(-1)
            df_sorted['penurunan_lift'] = df_sorted['Lift'] - df_sorted['lift_berikutnya']
            
            max_drop_index = df_sorted['penurunan_lift'].idxmax()
            max_drop_value = df_sorted['penurunan_lift'].max()
            
            print(f"   -> Titik potong (drop terbesar) awal di aturan ke-{max_drop_index + 1} (Gap: {max_drop_value:.4f})")
            
            # --- REVISI LOGIKA PENGAMAN (SAFETY NET) MINIMAL 9 KRITERIA ---
            kriteria_set = set()
            selected_index_limit = 0
            
            for i, row in df_sorted.iterrows():
                # Ekstrak kriteria untuk baris ini
                items = [item.strip() for item in str(row['Antecedents']).split(',')]
                kriteria_set.update(items)
                selected_index_limit = i
                
                # Cek berhenti: Jika sudah di/melewati titik potong DAN kriteria unik sudah >= 9
                if i >= max_drop_index and len(kriteria_set) >= 9:
                    break
            
            # Notifikasi jika batas diperluas
            if selected_index_limit > max_drop_index:
                print(f"   -> [REVISI AKTIF] Kriteria unik pada titik potong awal belum mencapai target.")
                print(f"   -> Pemotongan otomatis diperluas hingga aturan ke-{selected_index_limit + 1} untuk mengamankan {len(kriteria_set)} kriteria unik.")
            
            # Potong dataframe sesuai batas akhir yang sudah divalidasi
            df_selected = df_sorted.iloc[:selected_index_limit + 1].copy()
            
        else:
            print("   -> Hanya terdapat 1 aturan valid, langsung dipilih tanpa pemotongan.")
            df_selected = df_sorted.copy()
            
        print(f"   -> Total aturan signifikan yang diekstraksi: {len(df_selected)} aturan.")
        
        # 4. MENGEKSTRAKSI & MENGHITUNG FAKTOR RISIKO DOMINAN
        kriteria_list = []
        
        for index, row in df_selected.iterrows():
            items = [item.strip() for item in str(row['Antecedents']).split(',')]
            kriteria_list.extend(items)
            
        kriteria_counts = Counter(kriteria_list)
        
        df_kriteria = pd.DataFrame.from_dict(kriteria_counts, orient='index', columns=['Frekuensi_Kemunculan']).reset_index()
        df_kriteria = df_kriteria.rename(columns={'index': 'Faktor_Risiko'})
        df_kriteria = df_kriteria.sort_values(by='Frekuensi_Kemunculan', ascending=False).reset_index(drop=True)
        
        print(f"\n   Top Faktor Risiko Dominan untuk {tipe} - {target}:")
        # Jika kriteria lebih dari 10, tampilkan sesuai jumlah maksimal yang didapat agar 9 kriteria tercover
        display(df_kriteria.head(max(10, len(df_kriteria)))) 
        
        # 5. SIMPAN HASIL KE CSV
        nama_file = f'kriteria_dominan_{tipe}_{target}.csv'
        df_kriteria.to_csv(nama_file, index=False)
        print(f"   [✓] Hasil diekstrak ke: '{nama_file}'")

print("\n" + "="*80)
print(" SELURUH PROSES EKSTRAKSI SELESAI!")
print("="*80)

MEMULAI EKSEKUSI PEMOTONGAN DINAMIS & EKSTRAKSI KRITERIA (IT & R&D)
 [✓] File dimuat: hasil_fpgrowth_IT_sup_0_061_conf_0_5.csv (10476 aturan)
 [✓] File dimuat: hasil_fpgrowth_RnD_sup_0_061_conf_0_5.csv (37324 aturan)

######################################################################
 MENGANALISIS DOMAIN: PROYEK IT
######################################################################

--------------------------------------------------
 Target Risiko: Risk_Level_Critical
--------------------------------------------------
   -> Titik potong (drop terbesar) awal di aturan ke-2 (Gap: 0.3413)
   -> [REVISI AKTIF] Kriteria unik pada titik potong awal belum mencapai target.
   -> Pemotongan otomatis diperluas hingga aturan ke-5 untuk mengamankan 9 kriteria unik.
   -> Total aturan signifikan yang diekstraksi: 5 aturan.

   Top Faktor Risiko Dominan untuk IT - Risk_Level_Critical:


,Faktor_Risiko,Frekuensi_Kemunculan
0,Tech_Environment_Stability_Legacy/Unstable,4
1,Data_Security_Requirements_Strict,3
2,External_Dependencies_Count_External_Dependenc...,2
3,Team_Experience_Level_Junior,1
4,Budget_Utilization_Rate_Budget_Utilization_Rat...,1
5,Project_Budget_USD_Project_Budget_USD_High,1
6,Estimated_Timeline_Months_Estimated_Timeline_M...,1
7,Org_Process_Maturity_Ad-hoc,1
8,Technology_Familiarity_New,1


   [✓] Hasil diekstrak ke: 'kriteria_dominan_IT_Risk_Level_Critical.csv'

--------------------------------------------------
 Target Risiko: Risk_Level_High
--------------------------------------------------
   -> (BYPASS KHUSUS) Target Risk_Level_High terdeteksi.
   -> Mengambil seluruh aturan yang tersedia tanpa pemotongan.
   -> Total aturan signifikan yang diekstraksi: 50 aturan.

   Top Faktor Risiko Dominan untuk IT - Risk_Level_High:


,Faktor_Risiko,Frekuensi_Kemunculan
0,Risk_Management_Maturity_Basic,14
1,Team_Experience_Level_Mixed,13
2,Tech_Environment_Stability_Legacy/Unstable,11
3,Data_Security_Requirements_High,7
4,Org_Process_Maturity_Ad-hoc,6
5,Documentation_Quality_Poor,6
6,Change_Request_Frequency_Change_Request_Freque...,6
7,Complexity_Score_Complexity_Score_High,6
8,Technical_Debt_Level_Technical_Debt_Level_High,5
9,Org_Process_Maturity_Managed,4


   [✓] Hasil diekstrak ke: 'kriteria_dominan_IT_Risk_Level_High.csv'

--------------------------------------------------
 Target Risiko: Risk_Level_Low
--------------------------------------------------
   -> Titik potong (drop terbesar) awal di aturan ke-2 (Gap: 0.0912)
   -> [REVISI AKTIF] Kriteria unik pada titik potong awal belum mencapai target.
   -> Pemotongan otomatis diperluas hingga aturan ke-5 untuk mengamankan 9 kriteria unik.
   -> Total aturan signifikan yang diekstraksi: 5 aturan.

   Top Faktor Risiko Dominan untuk IT - Risk_Level_Low:


,Faktor_Risiko,Frekuensi_Kemunculan
0,Org_Process_Maturity_Optimizing,4
1,Team_Experience_Level_Expert,2
2,Key_Stakeholder_Availability_Excellent,2
3,Complexity_Score_Complexity_Score_Low,2
4,Change_Control_Maturity_Advanced,1
5,Regulatory_Compliance_Level_Medium,1
6,Tech_Environment_Stability_Modern/Stable,1
7,Estimated_Timeline_Months_Estimated_Timeline_M...,1
8,Stakeholder_Engagement_Level_Excellent,1


... (7 outputs left out)

,Faktor_Risiko,Frekuensi_Kemunculan
0,Project_Manager_Experience_Certified PM,108
1,Risk_Management_Maturity_Advanced,104
2,Change_Control_Maturity_Advanced,96
3,Industry_Volatility_Stable,66
4,Team_Experience_Level_Expert,63
...,...,...
107,Executive_Sponsorship_Weak,1
108,Budget_Utilization_Rate_Budget_Utilization_Rat...,1
109,Key_Stakeholder_Availability_Moderate,1
110,Contract_Type_Time & Materials,1


   [✓] Hasil diekstrak ke: 'kriteria_dominan_RnD_Risk_Level_Low.csv'

--------------------------------------------------
 Target Risiko: Risk_Level_Medium
--------------------------------------------------
   -> Titik potong (drop terbesar) awal di aturan ke-2 (Gap: 0.3738)
   -> [REVISI AKTIF] Kriteria unik pada titik potong awal belum mencapai target.
   -> Pemotongan otomatis diperluas hingga aturan ke-5 untuk mengamankan 10 kriteria unik.
   -> Total aturan signifikan yang diekstraksi: 5 aturan.

   Top Faktor Risiko Dominan untuk RnD - Risk_Level_Medium:


,Faktor_Risiko,Frekuensi_Kemunculan
0,Org_Process_Maturity_Optimizing,4
1,Contract_Type_Time & Materials,3
2,Organizational_Change_Frequency_Organizational...,1
3,Requirement_Stability_Stable,1
4,Past_Similar_Projects_Past_Similar_Projects_Me...,1
5,Resource_Contention_Level_Low,1
6,Regulatory_Compliance_Level_Medium,1
7,Team_Size_Team_Size_High,1
8,Change_Control_Maturity_Advanced,1
9,Funding_Source_Internal,1


   [✓] Hasil diekstrak ke: 'kriteria_dominan_RnD_Risk_Level_Medium.csv'

 SELURUH PROSES EKSTRAKSI SELESAI!
